# LLM Judging Runner

Use this notebook to run judging in two modes:
- `single_row`: fast sanity check on one row
- `full_df`: run judging on the full dataframe

<div class="alert alert-block alert-info">
<b>Note:</b> This notebook was used to improve the judging code and LLM prompt, as here we could isolate the judging process.
</div>

<div class="alert alert-block alert-warning">
<b>Part 6:</b> This notebook Tackles the entire part 6 in addition to what we did in previous notebooks.
</div>

In [10]:
import os
import pandas as pd

from llm_client import get_client
from llm_judge import run_judge
import numpy as np

In [ ]:
import os
os.environ["NEBIUS_API_KEY"] = "Your_key"

## Pipeline for single row or full dataframe

In [ ]:
MODE = 'full_df'  # options: 'single_row' or 'full_df'
ROW_INDEX = 26

INPUT_EXCEL_PATH = 'data/generated/ecommerce_sheet_new.xlsx'
OUTPUT_EXCEL_FULL = 'data/generated/ecommerce_sheet_new_judged_full.xlsx'
OUTPUT_EXCEL_SINGLE = f'data/generated/ecommerce_sheet_new_judged_row_{ROW_INDEX}.xlsx'

In [4]:
results_df = pd.read_excel(INPUT_EXCEL_PATH)
client = get_client()

if MODE == 'full_df':
    judged_df, judged_full_df = run_judge(
        results_df=results_df.copy(),
        client=client,
        excel_path=OUTPUT_EXCEL_FULL,
    )
    print(f'Judged full dataframe with {len(judged_df)} rows -> {OUTPUT_EXCEL_FULL}')

elif MODE == 'single_row':
    if not (0 <= ROW_INDEX < len(results_df)):
        raise IndexError(f'ROW_INDEX={ROW_INDEX} is out of range for dataframe size {len(results_df)}')

    single_row_df = results_df.iloc[[ROW_INDEX]].copy()
    judged_df, judged_full_df = run_judge(
        results_df=single_row_df,
        client=client,
        excel_path=OUTPUT_EXCEL_SINGLE,
    )
    print(f'Judged one row (ROW_INDEX={ROW_INDEX}) -> {OUTPUT_EXCEL_SINGLE}')

else:
    raise ValueError("MODE must be either 'single_row' or 'full_df'")

Task 5: Judging:   0%|          | 0/50 [00:00<?, ?it/s]

Judged full dataframe with 50 rows -> ecommerce_sheet_new_judged_full.xlsx


In [5]:
judged_full_df

,product_name,Product_attribute_list,material,warranty,generated_description,latency_ms,input_tokens,output_tokens,fluency,grammar,...,length,grounding,latency_rating,cost_rating,final_score,fluency_explanation,grammar_explanation,tone_explanation,length_explanation,grounding_explanation
0,Apple iPhone 15 Pro,"features: A17 Pro chip, 120 Hz ProMotion displ...","titanium frame, Ceramic Shield glass",1‑year limited warranty,"Enjoy lightning-fast performance and a smooth,...",4888,462,89,good,good,...,good,good,ok,good,pass,"The description reads naturally, with a clear ...",The description is free of spelling and punctu...,"The tone is generally good, as it addresses th...",I counted 76 words in the description.,Step A: Extract required facts from SOURCE DAT...
1,Samsung Galaxy S24 Ultra,"features: 200 MP camera, S‑Pen support, 120 Hz...","Armor Aluminum frame, Gorilla Glass Victus",1‑year limited warranty,Imagine capturing life's moments in stunning d...,896,462,100,good,good,...,good,good,good,good,pass,"The description reads naturally, with a clear ...",The description is free of spelling and punctu...,"The tone is conversational and engaging, with ...",I counted 76 words in the description.,Step A: Extract required facts from SOURCE DAT...
2,Google Pixel 8 Pro,"features: Tensor G3 chip, Magic Eraser, 50 MP ...","matte glass back, aluminum frame",1‑year limited warranty,Enjoy stunning photos and effortless editing w...,568,459,65,ok,good,...,good,bad,good,good,fail,"The description reads naturally, but there's a...",There are no spelling or punctuation errors in...,"The description has a good tone, as it address...",I counted 56 words in the description.,Step A: Extract required facts from SOURCE DAT...
3,Sony WH‑1000XM5 Headphones,"features: active noise cancelling, 30 hr batte...",synthetic leather earcups,1‑year limited warranty,Imagine tuning out the world and fully immersi...,813,459,98,ok,good,...,ok,good,good,good,pass,"The description reads naturally, with a clear ...",There are no spelling or punctuation errors in...,"The description has a good tone, as it address...",I counted 76 words in the description.,Step A: required_facts = ['active noise cancel...
4,Bose QuietComfort Ultra Earbuds,"features: CustomTune sound calibration, ANC, I...",silicone ear tips,1‑year limited warranty,Enjoy your audio in total harmony with the Bos...,763,448,90,ok,good,...,ok,good,good,good,pass,"The description reads naturally, with a clear ...",There are no spelling or punctuation errors in...,"The description has a good tone, as it address...",I counted 76 words in the description.,Step A: Extract required facts from SOURCE DAT...
5,Amazon Echo Dot (5th Gen),"features: Alexa voice assistant, temperature s...",recycled fabric mesh,1‑year limited warranty,Enjoy seamless control over your smart home wi...,718,448,80,ok,good,...,good,good,good,good,pass,"The description reads naturally, but there are...",There are no spelling or punctuation errors in...,The description uses some corporate filler phr...,I counted 69 words in the description.,Step A: Extract required facts from SOURCE DAT...
6,Dell XPS 13 9310 Laptop,"features: 13.4″ FHD+ display, Intel Core i5, T...",CNC aluminum & carbon‑fiber palmrest,1‑year limited hardware warranty,Enjoy effortless productivity with the Dell XP...,844,468,97,good,good,...,ok,good,good,good,pass,"The description reads naturally, with a clear ...",The description has zero spelling or punctuati...,"The tone is professional and informative, addr...",I counted 76 words in the description.,Step A: Extract required facts from SOURCE DAT...
7,Apple MacBook Air 13″ (M3),"features: M3 chip, 18‑hour battery, 1080p came...",recycled aluminum unibody,1‑year limited warranty,Imagine enjoying 18 hours of work or play on a...,1009,462,100,ok,good,...,good,good,good,good,pass,"The description reads naturally, with a clear ...",There are no spelling or punctuation errors in...,"The description has a good tone, as it add

***

In [6]:
judged_full_df.to_excel(f'ecommerce_sheet_new_judged_full.xlsx')

In [16]:
judged_full_df.to_excel(f'ecommerce_sheet_new_judged_row_{ROW_INDEX}_full.xlsx', index=False)

## Grounding-specific judge

In [5]:
from llm_judge import judge_one_metric
from tqdm.auto import tqdm


# Choose one metric to run separately: 'fluency', 'grammar', 'tone', 'grounding'
SPECIFIC_METRIC = "grounding"
client = get_client()

# Use judged_full_df if available, otherwise fallback to results_df
base_df = pd.read_excel(INPUT_EXCEL_PATH) 

df_separate = base_df.copy()
verdict_col = f"{SPECIFIC_METRIC}_separate_verdict"
explanation_col = f"{SPECIFIC_METRIC}_separate_explanation"

df_separate[verdict_col] = ""
df_separate[explanation_col] = ""

for idx, row in tqdm(df_separate.iterrows(), total=len(df_separate)):
    metric_result = judge_one_metric(client=client, row=row, metric=SPECIFIC_METRIC)
    df_separate.at[idx, verdict_col] = metric_result.verdict
    df_separate.at[idx, explanation_col] = metric_result.explanation

output_path = f"ecommerce_sheet_new_{SPECIFIC_METRIC}_separate.xlsx"
df_separate.to_excel(output_path, index=False)
print(f"Saved separate metric results to: {output_path}")

df_separate.head()

  0%|          | 0/50 [00:00<?, ?it/s]

Saved separate metric results to: ecommerce_sheet_new_grounding_separate.xlsx


,product_name,Product_attribute_list,material,warranty,generated_description,latency_ms,input_tokens,output_tokens,fluency,grammar,tone,length,grounding,latency_rating,cost_rating,final_score,grounding_separate_verdict,grounding_separate_explanation
0,Apple iPhone 15 Pro,"features: A17 Pro chip, 120 Hz ProMotion displ...","titanium frame, Ceramic Shield glass",1‑year limited warranty,"Enjoy a seamless, responsive experience with t...",1165,768,98,ok,good,ok,good,bad,good,good,fail,good,Grounding evaluation for Apple iPhone 15 Pro
1,Samsung Galaxy S24 Ultra,"features: 200 MP camera, S‑Pen support, 120 Hz...","Armor Aluminum frame, Gorilla Glass Victus",1‑year limited warranty,Capture every detail with the incredible 200MP...,918,768,96,ok,good,ok,good,good,good,good,pass,good,Grounding evaluation for Samsung Galaxy S24 Ultra
2,Google Pixel 8 Pro,"features: Tensor G3 chip, Magic Eraser, 50 MP ...","matte glass back, aluminum frame",1‑year limited warranty,Imagine capturing stunning photos with a 50MP ...,715,783,80,ok,good,ok,good,ok,good,good,pass,good,Grounding evaluation for Google Pixel 8 Pro
3,Sony WH‑1000XM5 Headphones,"features: active noise cancelling, 30 hr batte...",synthetic leather earcups,1‑year limited warranty,Enjoy immersive sound experiences with the Son...,826,765,104,good,good,good,ok,good,good,good,pass,good,Grounding evaluation for Sony WH-1000XM5 Headp...
4,Bose QuietComfort Ultra Earbuds,"features: CustomTune sound calibration, ANC, I...",silicone ear tips,1‑year limited warranty,Imagine enjoying your music without any distra...,808,754,86,ok,good,ok,ok,good,good,good,pass,good,Grounding evaluation for Bose QuietComfort Ult...


## Tone-specific judge

In [ ]:
from llm_judge import judge_one_metric
from tqdm.auto import tqdm

# Choose one metric to run separately: 'fluency', 'grammar', 'tone', 'grounding'
SPECIFIC_METRIC = "tone"
client = get_client()

# Use judged_full_df if available, otherwise fallback to results_df
base_df = pd.read_excel(INPUT_EXCEL_PATH) 

df_separate = base_df.copy()
verdict_col = f"{SPECIFIC_METRIC}_separate_verdict"
explanation_col = f"{SPECIFIC_METRIC}_separate_explanation"

df_separate[verdict_col] = ""
df_separate[explanation_col] = ""

for idx, row in tqdm(df_separate.iterrows(), total=len(df_separate)):
    metric_result = judge_one_metric(client=client, row=row, metric=SPECIFIC_METRIC)
    df_separate.at[idx, verdict_col] = metric_result.verdict
    df_separate.at[idx, explanation_col] = metric_result.explanation

output_path = f"ecommerce_sheet_new_{SPECIFIC_METRIC}_separate.xlsx"



  0%|          | 0/50 [00:00<?, ?it/s]

PermissionError: [Errno 13] Permission denied: 'ecommerce_sheet_new_tone_separate.xlsx'

In [ ]:
df_separate.to_excel(output_path, index=False)
print(f"Saved separate metric results to: {output_path}")


Saved separate metric results to: ecommerce_sheet_new_tone_separate.xlsx


In [6]:
df_separate.head()

,product_name,Product_attribute_list,material,warranty,generated_description,latency_ms,input_tokens,output_tokens,fluency,grammar,tone,length,grounding,latency_rating,cost_rating,final_score,tone_separate_verdict,tone_separate_explanation
0,Apple iPhone 15 Pro,"features: A17 Pro chip, 120 Hz ProMotion displ...","titanium frame, Ceramic Shield glass",1‑year limited warranty,"Enjoy a seamless, responsive experience with t...",1165,768,98,ok,good,ok,good,bad,good,good,fail,ok,The description uses buzz words ('seamless') a...
1,Samsung Galaxy S24 Ultra,"features: 200 MP camera, S‑Pen support, 120 Hz...","Armor Aluminum frame, Gorilla Glass Victus",1‑year limited warranty,Capture every detail with the incredible 200MP...,918,768,96,ok,good,ok,good,good,good,good,pass,ok,The description uses buzz words ('incredible')...
2,Google Pixel 8 Pro,"features: Tensor G3 chip, Magic Eraser, 50 MP ...","matte glass back, aluminum frame",1‑year limited warranty,Imagine capturing stunning photos with a 50MP ...,715,783,80,ok,good,ok,good,ok,good,good,pass,good,"The description uses a conversational tone, ad..."
3,Sony WH‑1000XM5 Headphones,"features: active noise cancelling, 30 hr batte...",synthetic leather earcups,1‑year limited warranty,Enjoy immersive sound experiences with the Son...,826,765,104,good,good,good,ok,good,good,good,pass,ok,The description addresses the customer directl...
4,Bose QuietComfort Ultra Earbuds,"features: CustomTune sound calibration, ANC, I...",silicone ear tips,1‑year limited warranty,Imagine enjoying your music without any distra...,808,754,86,ok,good,ok,ok,good,good,good,pass,ok,The description uses a benefit-focused opening...


## Fluency-specific judge

In [4]:
from llm_judge import judge_one_metric
from tqdm.auto import tqdm

# Choose one metric to run separately: 'fluency', 'grammar', 'tone', 'grounding'
SPECIFIC_METRIC = "fluency"
client = get_client()

# Use judged_full_df if available, otherwise fallback to results_df
base_df = pd.read_excel(INPUT_EXCEL_PATH) 

df_separate = base_df.copy()
verdict_col = f"{SPECIFIC_METRIC}_separate_verdict"
explanation_col = f"{SPECIFIC_METRIC}_separate_explanation"

df_separate[verdict_col] = ""
df_separate[explanation_col] = ""

for idx, row in tqdm(df_separate.iterrows(), total=len(df_separate)):
    metric_result = judge_one_metric(client=client, row=row, metric=SPECIFIC_METRIC)
    df_separate.at[idx, verdict_col] = metric_result.verdict
    df_separate.at[idx, explanation_col] = metric_result.explanation

output_path = f"ecommerce_sheet_new_{SPECIFIC_METRIC}_separate.xlsx"



  0%|          | 0/50 [00:00<?, ?it/s]

In [5]:
df_separate.to_excel(output_path, index=False)
print(f"Saved separate metric results to: {output_path}")


Saved separate metric results to: ecommerce_sheet_new_fluency_separate.xlsx


In [6]:
df_separate.head()

,product_name,Product_attribute_list,material,warranty,generated_description,latency_ms,input_tokens,output_tokens,fluency,grammar,tone,length,grounding,latency_rating,cost_rating,final_score,fluency_separate_verdict,fluency_separate_explanation
0,Apple iPhone 15 Pro,"features: A17 Pro chip, 120 Hz ProMotion displ...","titanium frame, Ceramic Shield glass",1‑year limited warranty,"Enjoy a seamless, responsive experience with t...",1165,768,98,ok,good,ok,good,bad,good,good,fail,good,The description reads naturally and flows well...
1,Samsung Galaxy S24 Ultra,"features: 200 MP camera, S‑Pen support, 120 Hz...","Armor Aluminum frame, Gorilla Glass Victus",1‑year limited warranty,Capture every detail with the incredible 200MP...,918,768,96,ok,good,ok,good,good,good,good,pass,good,The description reads naturally and flows well...
2,Google Pixel 8 Pro,"features: Tensor G3 chip, Magic Eraser, 50 MP ...","matte glass back, aluminum frame",1‑year limited warranty,Imagine capturing stunning photos with a 50MP ...,715,783,80,ok,good,ok,good,ok,good,good,pass,good,The description reads naturally and flows well...
3,Sony WH‑1000XM5 Headphones,"features: active noise cancelling, 30 hr batte...",synthetic leather earcups,1‑year limited warranty,Enjoy immersive sound experiences with the Son...,826,765,104,good,good,good,ok,good,good,good,pass,good,"The description flows naturally and logically,..."
4,Bose QuietComfort Ultra Earbuds,"features: CustomTune sound calibration, ANC, I...",silicone ear tips,1‑year limited warranty,Imagine enjoying your music without any distra...,808,754,86,ok,good,ok,ok,good,good,good,pass,good,The description reads naturally and flows well...


## Grammar-specific judgme

In [7]:
from llm_judge import judge_one_metric
from tqdm.auto import tqdm

# Choose one metric to run separately: 'fluency', 'grammar', 'tone', 'grounding'
SPECIFIC_METRIC = "grammar"
client = get_client()

# Use judged_full_df if available, otherwise fallback to results_df
base_df = pd.read_excel(INPUT_EXCEL_PATH) 

df_separate = base_df.copy()
verdict_col = f"{SPECIFIC_METRIC}_separate_verdict"
explanation_col = f"{SPECIFIC_METRIC}_separate_explanation"

df_separate[verdict_col] = ""
df_separate[explanation_col] = ""

for idx, row in tqdm(df_separate.iterrows(), total=len(df_separate)):
    metric_result = judge_one_metric(client=client, row=row, metric=SPECIFIC_METRIC)
    df_separate.at[idx, verdict_col] = metric_result.verdict
    df_separate.at[idx, explanation_col] = metric_result.explanation

output_path = f"ecommerce_sheet_new_{SPECIFIC_METRIC}_separate.xlsx"
df_separate.to_excel(output_path, index=False)
print(f"Saved separate metric results to: {output_path}")



  0%|          | 0/50 [00:00<?, ?it/s]

Saved separate metric results to: ecommerce_sheet_new_grammar_separate.xlsx


## Comparing multi-metric prompt to metric specific prompt and analyzing with human judgement

In [ ]:
full_df_path = 'data/generated/assignment_01_full_new.xlsx'
full_df = pd.read_excel(full_df_path)
grounding_specific_df = pd.read_excel('data/generated/ecommerce_sheet_new_grounding_separate.xlsx')
tone_specific_df = pd.read_excel('data/generated/ecommerce_sheet_new_tone_separate.xlsx')
fluency_specific_df = pd.read_excel('data/generated/ecommerce_sheet_new_fluency_separate.xlsx')
grammar_specific_df = pd.read_excel('data/generated/ecommerce_sheet_new_grammar_separate.xlsx')

In [14]:
sample_df_idx = full_df.sample(15, random_state=42).index
sample_full_df = full_df.loc[sample_df_idx]
grounding_specific_sample_df = grounding_specific_df.loc[sample_df_idx]
tone_specific_sample_df = tone_specific_df.loc[sample_df_idx]
fluency_specific_sample_df = fluency_specific_df.loc[sample_df_idx]
grammar_specific_sample_df = grammar_specific_df.loc[sample_df_idx]


In [16]:
sample_full_df

,product_name,Product_attribute_list,material,warranty,generated_description,latency_ms,input_tokens,output_tokens,fluency,grammar,...,grounding,latency_rating,cost_rating,final_score,fluency_explanation,grammar_explanation,tone_explanation,length_explanation,grounding_explanation,cost
13,Nintendo Switch OLED,"features: 7″ OLED screen, docked & handheld mo...",plastic chassis,1‑year limited warranty,Enjoy vibrant colors and stunning visuals with...,656,757,82,ok,good,...,bad,good,good,fail,"The description reads naturally, but there are...",There are no spelling or punctuation errors in...,The description uses some corporate filler wor...,I counted 76 words in the description.,Step A: Extract required facts from SOURCE DAT...,0.000020
39,Blink Outdoor 4,"features: 1080p HD, 2‑year battery, weather‑re...",plastic,1‑year limited warranty,Enjoy crystal-clear 1080p HD video footage of ...,672,756,79,good,good,...,good,good,good,pass,"The description reads naturally, with no awkwa...",There are no spelling or punctuation errors in...,"The tone is neutral and informative, without a...",I counted 66 words in the description.,Step A: Extract required facts from SOURCE DAT...,0.000020
30,LEGO Star Wars Millennium Falcon 75192,"features: 7 541 pieces, detailed interior, min...",ABS plastic bricks,lifetime replacement of missing bricks,"Imagine building the iconic Millennium Falcon,...",1017,764,86,ok,good,...,good,good,good,pass,"The description reads naturally, with a clear ...",There are no spelling or punctuation errors in...,"The description has a good tone, as it address...",I counted 76 words in the description.,Step A: Extract required facts from SOURCE DAT...,0.000020
45,SanDisk Extreme PRO 128 GB SDXC,"features: UHS‑I U3, 200 MB/s read, waterproof;...",plastic,lifetime limited warranty,You'll enjoy capturing every moment with the S...,800,765,98,good,good,...,good,good,good,pass,"The description reads naturally, with a clear ...",The description is free of spelling and punctu...,"The description has a conversational tone, add...",I counted 76 words in the description.,Step A: Extract required facts from SOURCE DAT...,0.000021
17,Keurig K‑Supreme Plus Smart,"features: multistream extraction, BrewID, 78 o...",brushed stainless steel accents,1‑year limited warranty,"Enjoy perfectly brewed coffee, every time, wit...",811,759,84,good,good,...,good,good,good,pass,"The description reads naturally, with a clear ...",There are no spelling or punctuation errors in...,The tone of the description is conversational ...,I counted 76 words in the description.,Step A: Extract required facts from SOURCE DAT...,0.000020
48,BenQ PD2725U 27″ Monitor,"features: 4K UHD IPS, 98% P3, Thunderbolt 3; c...",metal stand,3‑year warranty,Want to experience your digital content in stu...,745,766,95,good,good,...,good,good,good,pass,"The description reads naturally, with no awkwa...",There are no spelling or punctuation errors in...,"The description has a good tone, as it address...",I counted 76 words in the description.,Step A: Extract required facts from SOURCE DAT...,0.000021
26,Adidas Ultraboost Light,"features: Light BOOST midsole, Primeknit+ uppe...",textile & synthetic,2‑year manufacturer warranty,Imagine running on clouds – that’s the feeling...,867,778,92,ok,good,...,good,good,good,pass,"The description reads naturally, with a clear ...",There are no spelling or punctuation errors in...,"The description has a good tone, as it address...",I counted 76 words in the description.,Step A: Extract required facts from SOURCE DAT...,0.000021
25,Nike Air Zoom Pegasus 40,"features: React foam, Zoom Air units, engineer...",textile & synthetic,2‑year manufacturer warranty,Experience the effortless energy return of the...,762,754,99,good,good,...,good,good,good,pass,"The description reads naturally, with no awkwa...",There are no spelling or punctuation errors in...,"The description has a good tone, as it address...",I counted 76 words in the description.,Step A

***

### Helper function

In [ ]:
def matching_judging(
    full_df: pd.DataFrame,
    specific_df: pd.DataFrame,
    metric_name: str
) -> pd.DataFrame:
    """
    Compares the verdicts in the full_df with the verdicts in the specific_df for the given metric.
    Returns a DataFrame with the original columns plus a new column 'matching_verdict' indicating if they match.
    """
    product_name_col = 'product_name'
    generated_description_col = 'generated_description'
    verdict_col_full = f"{metric_name}"
    col_explanation_full = f"{metric_name}_explanation"
    
    verdict_col_specific = f"{metric_name}_separate_verdict"
    col_explanation_specific = f"{metric_name}_separate_explanation"
    
    comparison_df = full_df[[verdict_col_full, product_name_col, generated_description_col]].copy()
    comparison_df['specific_verdict'] = specific_df[verdict_col_specific]
    comparison_df['matching_verdict'] = comparison_df[verdict_col_full] == comparison_df['specific_verdict']
    comparison_df['full_explanation'] = full_df[col_explanation_full]
    comparison_df['specific_explanation'] = specific_df[col_explanation_specific]
    
    return comparison_df[comparison_df['matching_verdict'] == False]

In [29]:
grounding_comparison_df = matching_judging(sample_full_df, grounding_specific_sample_df, 'grounding')
tone_comparison_df = matching_judging(sample_full_df, tone_specific_sample_df, 'tone')      
fluency_comparison_df = matching_judging(sample_full_df, fluency_specific_sample_df, 'fluency')
grammar_comparison_df = matching_judging(sample_full_df, grammar_specific_sample_df, 'grammar')

***

## Grounding metric

In [30]:
from IPython.display import HTML, display
display(HTML(grounding_comparison_df[['grounding','specific_verdict', 'full_explanation', 'specific_explanation']].to_html(escape=False, index=False)))

grounding,specific_verdict,full_explanation,specific_explanation
bad,good,"Step A: Extract required facts from SOURCE DATA. required_facts = ['7″ OLED screen', 'docked & handheld modes', '64 GB storage', 'large capacity', 'plastic chassis', '1‑year limited warranty']. rating_required = false. required_rating_value = null. Step B: Extract claimed facts from DESCRIPTION only. claimed_facts = ['7” OLED screen', 'seamlessly transition between docked and handheld modes', 'large 64GB storage capacity', 'durable plastic chassis', '1-year limited warranty']. Step C: Coverage/support checks. For each required_fact, mark COVERED or MISSING in DESCRIPTION. For each claimed_fact, mark SUPPORTED or UNSUPPORTED vs SOURCE. required_facts: ['7″ OLED screen' -> COVERED, 'docked & handheld modes' -> COVERED, '64 GB storage' -> COVERED, 'large capacity' -> COVERED, 'plastic chassis' -> COVERED, '1‑year limited warranty' -> COVERED]. claimed_facts: ['7” OLED screen' -> SUPPORTED, 'seamlessly transition between docked and handheld modes' -> UNSUPPORTED (no transition mentioned in SOURCE), 'large 64GB storage capacity' -> SUPPORTED, 'durable plastic chassis' -> UNSUPPORTED (SOURCE says 'durable' is not present), '1-year limited warranty' -> SUPPORTED]. Step D: Mandatory decision gates. Since 'seamlessly transition between docked and handheld modes' is UNSUPPORTED, verdict = bad.",Grounding evaluation for Nintendo Switch OLED


The explanation is a bit lacking for some reason (even though the prompt is identical in the grounding part), but indeed seemless does not seem like a "hallucinated fact"

***

## Tone Metric

In [31]:
display(HTML(tone_comparison_df[['product_name','generated_description','tone','specific_verdict', 'full_explanation', 'specific_explanation']].to_html(escape=False, index=False)))

product_name,generated_description,tone,specific_verdict,full_explanation,specific_explanation
Nintendo Switch OLED,"Enjoy vibrant colors and stunning visuals with the Nintendo Switch OLED's 7” OLED screen. Seamlessly transition between docked and handheld modes for ultimate gaming flexibility. With a large 64GB storage capacity, you'll have plenty of room for all your favorite games. The Nintendo Switch OLED, housed in a durable plastic chassis, comes with a 1-year limited warranty.",ok,bad,"The description uses some corporate filler words like 'seamless' and 'durable'. However, it also addresses the customer directly with 'you'll have plenty of room for all your favorite games'. The tone is mostly appropriate, but it could be more engaging.","The description uses buzz words ('seamlessly'), corporate filler ('durable plastic chassis'), and robotic openers ('Enjoy vibrant colors...'). It also uses unsupported superlatives ('ultimate gaming flexibility')."
LEGO Star Wars Millennium Falcon 75192,"Imagine building the iconic Millennium Falcon, brick by brick, with its detailed interior and all your favorite minifigs. The LEGO Star Wars Millennium Falcon 75192 lets you do just that with 7,541 pieces for a truly immersive build. Enjoy hours of creative fun knowing that your build is powered by long-lasting batteries and that LEGO will replace any missing bricks for a lifetime.",ok,good,"The description has a good tone, as it addresses the customer directly and opens with a benefit. The use of active verbs like 'building' and 'lets you do just that' is also a good signal. However, the description could be more specific and grounded, as it relies on some generic adjectives like 'iconic' and 'immersive'.","The description opens with a benefit ('Imagine building...'), uses active verbs ('lets you do just that'), and addresses the customer directly ('your'). It also provides specific details about the product, such as the number of pieces and the warranty. The tone is conversational and recommends the product as if from a knowledgeable friend."
SanDisk Extreme PRO 128 GB SDXC,"You'll enjoy capturing every moment with the SanDisk Extreme PRO 128GB SDXC. This card features UHS-I U3 technology and a 200MB/s read speed, so you can shoot high-quality video and photos without lag. Plus, its waterproof design means you can take it anywhere, knowing your adventures are protected. With a long-lasting battery and a lifetime limited warranty, you can focus on what matters most.",ok,good,"The description has a conversational tone, addressing the customer directly with 'you' and opening with a benefit ('enjoy capturing every moment'). The language is active and concrete, using verbs like 'shoot' and 'take.' However, the phrase 'what matters most' could be seen as a bit generic.","The description addresses the customer directly, opens with a benefit, and uses active verbs. It also provides specific details about the product's features and warranty. However, it uses the phrase 'game-changer' is not present, but the description is still engaging and informative."
Keurig K‑Supreme Plus Smart,"Enjoy perfectly brewed coffee, every time, with the Keurig K‑Supreme Plus Smart. Its multistream extraction technology and BrewID feature work together to meet your individual coffee preference. Plus, with a large 78oz reservoir and brushed stainless steel accents, this machine is as stylish as it is convenient. Backed by a 1-year limited warranty, you can brew with confidence.",good,ok,"The tone of the description is conversational and friendly. It addresses the customer directly and opens with a benefit. The language is active and concrete, and the description reads like a recommendation from a knowledgeable friend. There are no buzz words, unsupported superlatives, or corporate filler.","The description addresses the customer directly, opens with a benefit, and uses active verbs. It also provides specific details about the product's features and warranty. Howeve

**Nintendo Switch OLED**- The specific metric prompt did better and followed the instructions and gave the correct grade, even though 'Enjoy vibrant colors' does not sound robotic to me. 1-0

**LEGO Star Wars Millennium Falcon 75192**- I agree with the multi metric prompt judge, as truly immersive really sounds too generic. for the tone specific judge it seemed good. 1-1

**SanDisk Extreme PRO 128 GB SDXC**- It seems that the multi metri prompt overfit to the generic part which is unexpected.  "'what matters most' could be seen as a bit generic", um I don't think it's that generic. 2-1

**Keurig K‑Supreme Plus Smart**- "'perfectly brewed coffee, every time'" sounds like a normal sells pitch to me, I agree with the multi metric prompt here. 2-2

**BenQ PD2725U 27″ Monitor**- Multi metric prompt missed the word "seamless". 3-2

**Adidas Ultraboost Light**- "top-quality running experience' could be seen as a slight corporate filler." I agree with the multi metric prompt here.  3-3

**Dyson V15 Detect Vacuum**- "buzz words, such as 'power' and 'long-lasting performance'" Multi metric prompt was off the mark here. 4-3

**Sony WH‑1000XM5 Headphones**- Again, multi metric prompt missed the word "seamless". 5-3

Overall, The specific tone prompt did slightly better but both judges were acceptable.


***

## Fluency Metric

In [32]:
display(HTML(fluency_comparison_df[['product_name','generated_description','fluency','specific_verdict', 'full_explanation', 'specific_explanation']].to_html(escape=False, index=False)))

product_name,generated_description,fluency,specific_verdict,full_explanation,specific_explanation
Nintendo Switch OLED,"Enjoy vibrant colors and stunning visuals with the Nintendo Switch OLED's 7” OLED screen. Seamlessly transition between docked and handheld modes for ultimate gaming flexibility. With a large 64GB storage capacity, you'll have plenty of room for all your favorite games. The Nintendo Switch OLED, housed in a durable plastic chassis, comes with a 1-year limited warranty.",ok,good,"The description reads naturally, but there are a few slightly awkward phrases. For example, 'Enjoy vibrant colors and stunning visuals' is a bit of a generic opening sentence. Additionally, the transition between docked and handheld modes could be smoother. However, the overall flow is good.","The description reads naturally and flows well, with no awkward phrasing or illogical claims."
LEGO Star Wars Millennium Falcon 75192,"Imagine building the iconic Millennium Falcon, brick by brick, with its detailed interior and all your favorite minifigs. The LEGO Star Wars Millennium Falcon 75192 lets you do just that with 7,541 pieces for a truly immersive build. Enjoy hours of creative fun knowing that your build is powered by long-lasting batteries and that LEGO will replace any missing bricks for a lifetime.",ok,good,"The description reads naturally, with a clear and engaging tone. The sentences flow well, and there are no awkward or illogical phrasing. However, the transition from the first sentence to the second sentence could be smoother.","The description reads naturally and flows well, with no awkward phrasing or illogical claims."
Adidas Ultraboost Light,"Imagine running on clouds – that’s the feeling you get with the Adidas Ultraboost Light. The Light BOOST midsole and Primeknit+ upper work together to keep your feet light and comfortable mile after mile, while the Continental rubber outsole provides superior grip on any surface. With a 4.7/5 rating from runners who love these shoes, you can feel confident you’re choosing a top-quality running experience.",ok,good,"The description reads naturally, with a clear and consistent tone throughout. The sentences flow well, and the language is engaging. However, the transition from the first sentence to the second could be smoother.","The description reads naturally and effectively conveys the key features and benefits of the product. The language is engaging and evocative, with a clear and concise structure."
Dyson V15 Detect Vacuum,"Experience the power of Dyson's laser dust detection technology. You'll see dust you never knew existed, and the V15 Detect's HEPA filtration will capture it, leaving your home cleaner than ever. Enjoy a full 60 minutes of cleaning time on a single charge, powered by the energy-efficient motor. The lightweight polycarbonate and aluminum construction makes it easy to maneuver, so you can effortlessly tackle any mess. With a 2-year parts and labor warranty, you can trust in the Dyson V15 Detect's long-lasting performance.",ok,good,"The description reads naturally, with a clear and consistent tone throughout. The sentences flow well, and the transitions between ideas are smooth. However, there are a few slightly unnatural phrases, such as 'You'll see dust you never knew existed', which could be rephrased for better clarity.","The description reads naturally and flows well, with no awkward phrasing or illogical claims."
Bose QuietComfort Ultra Earbuds,"Imagine enjoying your music without any distractions, no matter where you are. Bose QuietComfort Ultra Earbuds use CustomTune sound calibration to create a personalized listening experience, while the industry-leading ANC technology blocks out unwanted noise. With an IPX4 rating for water resistance and award-winning design, these earbuds are built for comfort and performance. Plus, they come with silicone ear tips for a secure and comfortable fit.",ok,good,"The description reads naturally, but there are a few sligh

**Nintendo Switch OLED**- "is a bit of a generic opening sentence." The multi metric judge shows a classic leakage from tone to fluency. agree with fluenct specific judge. 1-0

**LEGO Star Wars Millennium Falcon 75192**- I tend to agree with the multi metric prompt here, "the transition from the first sentence to the second sentence could be smoother.", indeed. 1-1

**Adidas Ultraboost Light**- I see a pattern here for the multi metric prompt, "However, the transition from the first sentence to the second could be smoother.", tend to disagree but it might be different opinions now. 2-1

**Dyson V15 Detect Vacuum**- "'You'll see dust you never knew existed', " I agree with the multi metric prompt that's an odd phrase you don't 'see' the dust. 2-2

**Bose QuietComfort Ultra Earbuds**- "'no matter where you are' which feels like a transition phrase.", multi metric prompt judge here is a bit too highbrow here. 3-2

**TP‑Link Deco XE75 Mesh WiFi 6E**- a pattern here as well the multi metric judge has a problem with the classic phrase "No matter where you are" but in this explanation, "'will keep you connected no matter where you are.' This phrase is a bit of a non-sequitur and disrupts the flow of the sentence.", I tend to agree. 3-3

Overall,I think they're equal and they did ok job.


***

## Grammar Metric

In [33]:
display(HTML(grammar_comparison_df[['product_name','generated_description','grammar','specific_verdict', 'full_explanation', 'specific_explanation']].to_html(escape=False, index=False)))

product_name,generated_description,grammar,specific_verdict,full_explanation,specific_explanation
Nintendo Switch OLED,"Enjoy vibrant colors and stunning visuals with the Nintendo Switch OLED's 7” OLED screen. Seamlessly transition between docked and handheld modes for ultimate gaming flexibility. With a large 64GB storage capacity, you'll have plenty of room for all your favorite games. The Nintendo Switch OLED, housed in a durable plastic chassis, comes with a 1-year limited warranty.",good,ok,"There are no spelling or punctuation errors in the description. The grammar is correct, and the sentences are well-structured.","The description is clear and concise, but there are a few minor errors in punctuation and formatting."
Blink Outdoor 4,"Enjoy crystal-clear 1080p HD video footage of your home, day or night, with the Blink Outdoor 4. Its weather-resistant plastic design can withstand the elements, while the 2‑year battery life keeps it running reliably. This limited edition Blink Outdoor 4 comes with a 1‑year limited warranty for your added peace of mind.",good,ok,"There are no spelling or punctuation errors in the description. The grammar is correct, and the sentences are well-structured.","The description is clear and concise, but there are a few minor errors in punctuation and hyphenation."
LEGO Star Wars Millennium Falcon 75192,"Imagine building the iconic Millennium Falcon, brick by brick, with its detailed interior and all your favorite minifigs. The LEGO Star Wars Millennium Falcon 75192 lets you do just that with 7,541 pieces for a truly immersive build. Enjoy hours of creative fun knowing that your build is powered by long-lasting batteries and that LEGO will replace any missing bricks for a lifetime.",good,ok,"There are no spelling or punctuation errors in the description. The grammar is correct, and the sentences are well-structured.","The description is well-written and easy to understand, but there are a few minor errors in punctuation and formatting."
SanDisk Extreme PRO 128 GB SDXC,"You'll enjoy capturing every moment with the SanDisk Extreme PRO 128GB SDXC. This card features UHS-I U3 technology and a 200MB/s read speed, so you can shoot high-quality video and photos without lag. Plus, its waterproof design means you can take it anywhere, knowing your adventures are protected. With a long-lasting battery and a lifetime limited warranty, you can focus on what matters most.",good,ok,"The description is free of spelling and punctuation errors, making it easy to understand.","The description is clear and concise, but there are a few minor errors in punctuation and formatting."
BenQ PD2725U 27″ Monitor,"Want to experience your digital content in stunning clarity? The BenQ PD2725U 27″ Monitor delivers vibrant colors and sharp details with its 4K UHD IPS display and 98% P3 wide color gamut. Enjoy seamless connectivity with the Thunderbolt 3 port, and appreciate the sleek, modern design of the metal stand. Your creative projects will truly shine with this reliable monitor backed by a 3‑year warranty.",good,ok,"There are no spelling or punctuation errors in the description. The grammar is correct, and the sentences are well-structured.","The description is well-written and effectively communicates the product's features and benefits. However, there is a minor error in the punctuation of the warranty period."
Adidas Ultraboost Light,"Imagine running on clouds – that’s the feeling you get with the Adidas Ultraboost Light. The Light BOOST midsole and Primeknit+ upper work together to keep your feet light and comfortable mile after mile, while the Continental rubber outsole provides superior grip on any surface. With a 4.7/5 rating from runners who love these shoes, you can feel confident you’re choosing a top-quality running experience.",good,ok,"There are no spelling or punctuation errors in the description. The grammar is correct, and the sentences are well-structured.","The description is well-written and effectively 

The grammar specific judge has interesting takes here that are not according to the prompt about formatting and it seemed to hallucinate in Adidas Ultraboost Light. Admittedly, the prompt for grammar judging was a bit lacking and the multi metric judge gave all the rows good in grammar (Which I don't disagree with).

***

# Summary

**Analysis** 

a. LLM-as-a-judge will cost less, can be adjusted to way tlarger scales and maintain consistency. However without using the top-tier models and in harder tasks than the one we tackled the accuracy can be lower. 
b. combination of LLM-as-a-judge combined with coding guardrails is the better approach for thousands of daily descriptions.